# Guided investigation: how does layer 1 find its targets?

**Model:** the 2-layer attention-only transformer trained on in-context Markov chains with 20 symbols and length 256 (run `k20_L256_l2h1d64_s0_dense`; see `notes/decision_log.md`, entries P1–P10).

**The puzzle.** To implement the bigram strategy, the textbook circuit is:

1. **Layer 0, previous-token head:** each position copies in "my predecessor was X".
2. **Layer 1, statistical-induction head:** the current position (symbol *a*) attends to earlier positions whose *predecessor* was *a*, and reads the symbol there.

Step 2 seems to need step 1. Yet in training, layer 1's induction score jumped (0.08 → 0.41 between steps 2,250 and 2,500) while layer 0's previous-token score was still 0.05, and layer 0 kept strengthening for thousands of steps after the loss had nearly bottomed out.

**Your question:** what is layer 1 reading at step 2,500, and what changes by step 11,000?

### How to use this notebook
- Run top to bottom. Nothing here states the answer; the tools are set up and the questions are posed.
- Before each experiment there is a **✍️ Prediction** cell. Write your prediction *before* running the next cell, then compare. This is the predictions-first discipline from `notes/predictions.md`, at small scale.
- Every tool is checked in Part 1 before it is used. If a check fails, stop: the instrument is broken, not the model.
- Code lives in `src/toy_patching_compute.py` (tools) and `plotting/toy_plots.py` (figures). Read the docstrings; they say exactly what each measurement is.
- The checkpoints are git-ignored. If `data/toy/k20_L256_l2h1d64_s0_dense/` is missing, regenerate it with `python workflows/toy_markov_train.py --k 20 --length 256 --steps 11000 --dense-until 1000 --run-tag dense` (about 100 minutes).

In [ ]:
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd()
if not (_PROJECT_ROOT / "plotting").is_dir():
    _PROJECT_ROOT = _PROJECT_ROOT.parent
if not (_PROJECT_ROOT / "plotting").is_dir():
    raise RuntimeError(f"Cannot locate the project root from {Path.cwd()}. "
                       f"Start the kernel in the repository or in notebooks/.")
sys.path.insert(0, str(_PROJECT_ROOT / "src"))
sys.path.insert(0, str(_PROJECT_ROOT / "plotting"))

import matplotlib.pyplot as plt
import torch

from toy_patching_compute import (
    load_toy_run, load_checkpoint, make_eval_batch, sweep,
    run_metrics, previous_token_score, statistical_induction_score, attention_patterns,
    ablate, shuffle_within_sequences, patch_recovery,
    layer1_pattern_from_key_components, key_component_table,
)
from toy_plots import plot_training_stages, plot_attention_patterns, plot_metric_vs_step

torch.set_grad_enabled(False)
%matplotlib inline

RUN_DIR     = _PROJECT_ROOT / "data" / "toy" / "k20_L256_l2h1d64_s0_dense"
FIGURES_DIR = _PROJECT_ROOT / "figures" / "notebooks" / "toy_induction_guided"

In [ ]:
cfg, hist, steps = load_toy_run(RUN_DIR)
print(cfg)
print(f"{len(steps)} checkpoints, from step {steps[0]} to {steps[-1]}")

# A fresh, seeded evaluation batch (not the training-time eval set) and its exact
# reference log-probabilities (uniform / unigram / Bayes-optimal bigram).
x, refs = make_eval_batch(cfg, n=256, seed=12345)
x_corrupt = shuffle_within_sequences(x, seed=0)   # same symbols, order destroyed

### Recap: the training curves
The same figure as `figures/workflows/toy_markov_train/stages_k20_*.png`. Find the step-2,250 to 3,000 transition and note the order in which the two attention scores rise.

In [ ]:
fig = plot_training_stages(hist, "Recap: 2-layer attn-only, k=20, L=256")
plt.show()

---
## Part 1: Check the instruments

Three checks, on the final checkpoint:

1. `layer1_pattern_from_key_components` rebuilds layer 1's attention from the residual components. With **all** components included it should reproduce the model's own pattern.
2. `patch_recovery` runs the model on the shuffled input but overwrites one activation with its value from the clean run. Patching the **token embedding** (`hook_embed`) replaces the only thing that differs between the two inputs.
3. Patching the **positional embedding** (`hook_pos_embed`) replaces something that is identical in both runs.

### ✍️ Prediction
What value should each check give, and why? (Pattern error: about ___ ; recovered for `hook_embed`: ___ ; for `hook_pos_embed`: ___ )

*Your answer:*

In [ ]:
final = load_checkpoint(RUN_DIR, cfg, steps[-1])

P_model = attention_patterns(final, x)[1]
P_rebuilt = layer1_pattern_from_key_components(final, x, ("embed", "pos", "attn0"))
print("1. max |model pattern − rebuilt pattern| =", float((P_model - P_rebuilt).abs().max()))
print("2. recovered, patching hook_embed     =", patch_recovery(final, x, x_corrupt, "hook_embed")["recovered"])
print("3. recovered, patching hook_pos_embed =", patch_recovery(final, x, x_corrupt, "hook_pos_embed")["recovered"])

---
## Part 2: Look before you measure

Attention patterns for one sequence at four checkpoints: just before the transition (2,000), in the middle of it (2,500), just after it (3,000), and at the end (11,000). The left panel is layer 0, the right is layer 1. Token labels run along both axes.

### ✍️ Prediction
At step 2,500, will layer 0 look like a previous-token head (a strong line one step below the diagonal)? Will layer 1 already attend to "positions after earlier copies of the current symbol"?

*Your answer:*

In [ ]:
LOOK_STEPS = [2000, 2500, 3000, steps[-1]]
for s in LOOK_STEPS:
    m = load_checkpoint(RUN_DIR, cfg, s)
    pats = [p[0] for p in attention_patterns(m, x[:1])]
    fig = plot_attention_patterns(pats, x[0], f"Step {s}", n_show=40,
                                  save_path=FIGURES_DIR / f"attention_step{s:06d}.png")
    plt.show()

**Questions to answer in words** (a markdown cell below is fine):
- At step 2,500, what does layer 0 attend to, if not the previous token?
- Pick one query row in layer 1 at step 2,500. Which keys get the weight? Check their predecessors by hand against the query's symbol.
- What visibly changes between 3,000 and 11,000?

*Your notes:*

---
## Part 3: Does layer 1 need layer 0? (ablation)

`ablate(..., "blocks.0.hook_attn_out", "mean")` replaces layer 0's output at every position with its average over the batch at that position. Whatever layer 0 writes that is the same for every sequence survives; whatever is specific to *this* sequence (such as "my predecessor was X") is removed.

The metric is KL(bigram ‖ model): how far the model's predictions are from the Bayes-optimal strategy (0 = matches it).

### ✍️ Prediction
Sketch two curves against training step, from 1,500 to 11,000: normal, and with layer 0 mean-ablated. Where will they be close, and where far apart? What would each outcome mean for the puzzle?

*Your answer:*

In [ ]:
ABL_STEPS = [s for s in steps if s >= 1500]

def normal_and_ablated(m):
    return (run_metrics(m, x, refs)["kl_bigram"],
            ablate(m, x, refs, "blocks.0.hook_attn_out", "mean")["kl_bigram"])

res = sweep(RUN_DIR, cfg, ABL_STEPS, normal_and_ablated)
fig = plot_metric_vs_step(
    ABL_STEPS,
    {"Normal": [r[0] for r in res], "Layer 0 output mean-ablated": [r[1] for r in res]},
    ylabel="KL(bigram ‖ model), bits", title="Does the model still work without layer 0?",
    mark_steps=[2500], save_path=FIGURES_DIR / "ablate_layer0.png")
plt.show()

**Questions:**
- Is there a range of steps where removing layer 0 barely matters? A range where it matters a lot?
- Mean-ablation keeps the *position-specific* average of layer 0's output. Why might that matter here? (Look back at your Part 2 notes.)
- Try `mode="zero"` instead. Does the answer change? What does the difference between the two modes tell you?

---
## Part 4: What does layer 1's key read?

Layer 1 decides where to attend by comparing its **query** (at the current position) with a **key** at each earlier position. The key at a position is computed from the residual stream there, which is an exact sum of three parts:

| Component | Hook | Carries |
|---|---|---|
| `embed` | `hook_embed` | which symbol is at this position |
| `pos` | `hook_pos_embed` | which position this is |
| `attn0` | `blocks.0.hook_attn_out` | whatever layer 0's head wrote here |

`key_component_table` rebuilds layer 1's attention with keys made from only some of these parts, and reports the statistical-induction score (the baseline for uniform attention is about 0.05). The queries are left untouched.

### ✍️ Prediction
For each subset below, at step 2,500 and at step 11,000: will the induction score stay high, or collapse toward 0.05?

| Keys built from | Step 2,500 | Step 11,000 |
|---|---|---|
| all three | | |
| embed + pos (no layer 0) | | |
| embed + attn0 (no position) | | |
| pos + attn0 (no token identity) | | |

*Fill in the table.*

In [ ]:
SUBSETS = {
    "All three":                  ("embed", "pos", "attn0"),
    "No layer 0 (embed + pos)":   ("embed", "pos"),
    "No position (embed + attn0)": ("embed", "attn0"),
    "No token (pos + attn0)":     ("pos", "attn0"),
}
KEY_STEPS = [s for s in steps if s >= 1500]
tables = sweep(RUN_DIR, cfg, KEY_STEPS, lambda m: key_component_table(m, x, SUBSETS))
fig = plot_metric_vs_step(
    KEY_STEPS, {name: [t[name] for t in tables] for name in SUBSETS},
    ylabel="Layer 1 statistical-induction score", title="Which key components does layer 1 rely on?",
    mark_steps=[2500], save_path=FIGURES_DIR / "key_components.png")
plt.show()

**Questions:**
- Which component(s) does layer 1's key depend on at 2,500? At 11,000?
- If a component matters on its own, which earlier result (Part 2 or Part 3) does that agree with?
- This test changes only the **keys**. What would you change to ask the same question about the **queries**? (`layer1_pattern_from_key_components` is short; copying it into a query version is a good exercise.)

---
## Part 5: Where does the sequence-specific information flow? (activation patching)

Run the model on the shuffled sequence (same symbols, order destroyed), but overwrite one activation with its value from the clean run. `recovered` = 1 means that activation alone carries everything needed to restore the clean predictions; 0 means it carries none of it.

### ✍️ Prediction
Across training steps, how much does patching **layer 0's output** recover? And **layer 1's output** (`blocks.1.hook_attn_out`)? Why might neither be exactly 1?

*Your answer:*

In [ ]:
PATCH_STEPS = [s for s in steps if s >= 1500]

def recoveries(m):
    return (patch_recovery(m, x, x_corrupt, "blocks.0.hook_attn_out")["recovered"],
            patch_recovery(m, x, x_corrupt, "blocks.1.hook_attn_out")["recovered"])

rec = sweep(RUN_DIR, cfg, PATCH_STEPS, recoveries)
fig = plot_metric_vs_step(
    PATCH_STEPS,
    {"Patch layer 0 output": [r[0] for r in rec], "Patch layer 1 output": [r[1] for r in rec]},
    ylabel="Fraction of clean behavior recovered", title="Activation patching, clean → shuffled",
    mark_steps=[2500], save_path=FIGURES_DIR / "patch_recovery.png")
plt.show()

---
## Wrap-up: write your explanation

In three or four sentences: **how does layer 1 find its targets at step 2,500, and what changes by step 11,000?** Cite the part of this notebook each claim rests on.

*Your explanation:*

Then answer:
- **What result would have shown your explanation was wrong?** If you can't name one, the explanation isn't specific enough yet.
- **What is the cheapest next experiment that could break it?**

When you have an explanation you're willing to test, add it to `notes/predictions.md` *before* running that next experiment.

### Going further
- The ARENA mechanistic-interpretability material on induction heads and activation patching covers these same tools in depth, on GPT-2 and small attention-only models.
- The same notebook works on the 3-symbol runs: set `RUN_DIR` to `data/toy/k3_L100_l2h1d64_s0_dense`. The anomaly appeared there too.